In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "168c5e7c",
   "metadata": {},
   "source": [
    "# Urban Energy Analytics: NYC vs LA 2022\n",
    "\n",
    "## PCA + Clustering Algorithm Comparison for Residential Energy Consumption\n",
    "\n",
    "**Team:**\n",
    "- Atharva Prasanna Mokashi (SJSU ID: 019117046)\n",
    "- Maitreya Patankar (SJSU ID: 019146166)\n",
    "- Vineet Malewar (SJSU ID: 018399589)\n",
    "- Shefali Saini (SJSU ID: 018281848)\n",
    "\n",
    "---\n",
    "\n",
    "## Overview\n",
    "\n",
    "This notebook analyzes the relationship between socio-economic characteristics and residential electricity consumption across ZIP codes in NYC and LA using 2022 data from EIA Form 861 and ACS.\n",
    "\n",
    "**Pipeline:**\n",
    "1. Load EIA + ACS data\n",
    "2. Clean and integrate datasets\n",
    "3. Filter to NYC and LA\n",
    "4. Exploratory Data Analysis (EDA)\n",
    "5. Engineer 5 modeling features\n",
    "6. Apply PCA for dimensionality reduction\n",
    "7. **Compare three clustering algorithms: K-Means, Agglomerative Hierarchical, DBSCAN**\n",
    "8. **Select best algorithm based on Silhouette, Davies-Bouldin, and Calinski-Harabasz scores**\n",
    "9. Profile clusters and compare NYC vs LA patterns"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "f72f46c9",
   "metadata": {},
   "source": [
    "---\n",
    "\n",
    "## Section 0: Setup and Imports"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 33,
   "id": "d5e5f2a3",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "✓ All imports successful\n"
     ]
    }
   ],
   "source": [
    "# If running in Google Colab, uncomment and run:\n",
    "# from google.colab import drive\n",
    "# drive.mount('/content/drive')\n",
    "# !pip install -r requirements.txt\n",
    "\n",
    "import sys\n",
    "import os\n",
    "\n",
    "# Add src to path — works from notebooks/ or repo root\n",
    "_nb_dir = os.path.abspath(os.getcwd())\n",
    "_repo_root = _nb_dir if os.path.isdir(os.path.join(_nb_dir, 'src')) else os.path.dirname(_nb_dir)\n",
    "_src_path = os.path.join(_repo_root, 'src')\n",
    "if _src_path not in sys.path:\n",
    "    sys.path.insert(0, _src_path)\n",
    "\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from scipy.cluster.hierarchy import dendrogram, linkage\n",
    "\n",
    "from data_loader import load_eia_data, load_acs_data\n",
    "from data_cleaner import clean_and_integrate\n",
    "from feature_engineering import engineer_features, get_feature_matrix\n",
    "from modeling import (\n",
    "    standardize_features, apply_pca,\n",
    "    apply_clustering, apply_kmeans, apply_dbscan,\n",
    "    compare_algorithms, evaluate_clustering\n",
    ")\n",
    "\n",
    "plt.style.use('seaborn-v0_8-darkgrid')\n",
    "sns.set_palette('husl')\n",
    "plt.rcParams['figure.figsize'] = (12, 6)\n",
    "\n",
    "print('✓ All imports successful')"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "76524cf6",
   "metadata": {},
   "source": [
    "---\n",
    "\n",
    "## Section 1: Data Loading"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 34,
   "id": "df92ef29",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "[EIA] Loaded 39,075 ZIP codes from eia861_sales_2022.csv\n",
      "[EIA] Columns: ['ZIP', 'state', 'residential_mwh_sales', 'num_customers']\n",
      "[EIA] Shape: (39075, 4)\n",
      "[ACS] Loaded 29,996 ZCTAs from acs_zcta_2022.csv\n",
      "[ACS] Columns: ['ZIP', 'population', 'median_income', 'median_year_structure_built', 'total_occupied_units', 'renter_occupied_units']\n",
      "[ACS] Shape: (29996, 6)\n",
      "\n",
      "============================================================\n",
      "EIA Raw Data Head:\n",
      "============================================================\n",
      "     ZIP state  residential_mwh_sales  num_customers\n",
      "0  00501    NY               22466.24           2456\n",
      "1  00544    NY               22466.24           2456\n",
      "2  01002    MA               10617.27           1555\n",
      "3  01005    MA               10617.27           1555\n",
      "4  01007    MA               10617.27           1555\n",
      "\n",
      "============================================================\n",
      "ACS Raw Data Head:\n",
      "============================================================\n",
      "     ZIP  population  median_income  median_year_structure_built  \\\n",
      "0  00601       16834        17526.0                       1980.0   \n",
      "1  00602       37642        20260.0                       1978.0   \n",
      "2  00603       49075        17703.0                       1980.0   \n",
      "3  00606        5590        19603.0                       1978.0   \n",
      "4  00610       25542        22796.0                       1978.0   \n",
      "\n",
      "   total_occupied_units  renter_occupied_units  \n",
      "0                  5341                   1601  \n",
      "1                 12777                   3147  \n",
      "2                 19624                   8366  \n",
      "3                  1948                    485  \n",
      "4                  8781                   2389  \n"
     ]
    }
   ],
   "source": [
    "eia_raw = load_eia_data()\n",
    "acs_raw = load_acs_data()\n",
    "\n",
    "print(\"\\n\" + \"=\"*60)\n",
    "print(\"EIA Raw Data Head:\")\n",
    "print(\"=\"*60)\n",
    "print(eia_raw.head())\n",
    "\n",
    "print(\"\\n\" + \"=\"*60)\n",
    "print(\"ACS Raw Data Head:\")\n",
    "print(\"=\"*60)\n",
    "print(acs_raw.head())"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "6713209c",
   "metadata": {},
   "source": [
    "---\n",
    "\n",
    "## Section 2: Data Cleaning and Integration"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 35,
   "id": "d716dd67",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "============================================================\n",
      "STEP 1: Clean EIA Data\n",
      "============================================================\n",
      "[Clean EIA] After cleaning and aggregation: 39,075 unique ZIPs\n",
      "\n",
      "============================================================\n",
      "STEP 2: Clean ACS Data\n",
      "============================================================\n",
      "[Clean ACS] After cleaning: 29,996 ZCTAs retained\n",
      "\n",
      "============================================================\n",
      "STEP 3: Merge EIA + ACS\n",
      "============================================================\n",
      "[Merge] EIA ZIPs: 39,075, ACS ZCTAs: 29,996\n",
      "[Merge] Inner join result: 28,692 merged rows\n",
      "[Merge] Loss: 26.6% of EIA ZIPs (PO Box-only, non-ZCTA, etc.)\n",
      "\n",
      "============================================================\n",
      "STEP 4: Filter to NYC + LA\n",
      "============================================================\n",
      "[Filter] Extracted 476 ZIP codes for NYC and LA\n",
      "  NYC: 185\n",
      "  LA:  291\n",
      "\n",
      "============================================================\n",
      "INTEGRATION COMPLETE\n",
      "============================================================\n",
      "\n",
      "============================================================\n",
      "Integrated Dataset:\n",
      "============================================================\n",
      "Shape: (476, 10)\n",
      "\n",
      "Columns: ['ZIP', 'residential_mwh_sales', 'num_customers', 'state', 'population', 'median_income', 'median_year_structure_built', 'total_occupied_units', 'renter_occupied_units', 'city']\n",
      "\n",
      "First few rows:\n",
      "        ZIP  residential_mwh_sales  num_customers state  population  \\\n",
      "2193  10001               17799.76           3805    NY       27004   \n",
      "2194  10002               17799.76           3805    NY       76518   \n",
      "2195  10003               17799.76           3805    NY       53877   \n",
      "2196  10004               17799.76           3805    NY        4579   \n",
      "2197  10005               17799.76           3805    NY        8801   \n",
      "\n",
      "      median_income  median_year_structure_built  total_occupied_units  \\\n",
      "2193       106509.0                       1969.0                 14375   \n",
      "2194        43362.0                       1952.0                 36028   \n",
      "2195       152863.0                       1938.0                 24987   \n",
      "2196       232543.0                       1938.0                  2123   \n",
      "2197       189886.0                       1938.0                  4881   \n",
      "\n",
      "      renter_occupied_units city  \n",
      "2193                  10953  NYC  \n",
      "2194                  29804  NYC  \n",
      "2195                  16060  NYC  \n",
      "2196                   1230  NYC  \n",
      "2197                   4067  NYC  \n",
      "\n",
      "============================================================\n",
      "Summary by City:\n",
      "============================================================\n",
      "      population  residential_mwh_sales  num_customers\n",
      "city                                                  \n",
      "LA      10649638             9395487.58        1501440\n",
      "NYC      8650813             3505350.96         714138\n"
     ]
    }
   ],
   "source": [
    "df_integrated = clean_and_integrate(eia_raw, acs_raw)\n",
    "\n",
    "print(\"\\n\" + \"=\"*60)\n",
    "print(\"Integrated Dataset:\")\n",
    "print(\"=\"*60)\n",
    "print(f\"Shape: {df_integrated.shape}\")\n",
    "print(f\"\\nColumns: {df_integrated.columns.tolist()}\")\n",
    "print(f\"\\nFirst few rows:\")\n",
    "print(df_integrated.head())\n",
    "\n",
    "print(\"\\n\" + \"=\"*60)\n",
    "print(\"Summary by City:\")\n",
    "print(\"=\"*60)\n",
    "print(df_integrated.groupby('city')[['population', 'residential_mwh_sales', 'num_customers']].sum())"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a34faa22",
   "metadata": {},
   "source": [
    "---\n",
    "\n",
    "## Section 3: Exploratory Data Analysis (EDA)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 36,
   "id": "023030dc",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Descriptive Statistics:\n",
      "       residential_mwh_sales  num_customers     population  median_income\n",
      "count             476.000000     476.000000     476.000000     476.000000\n",
      "mean            27102.601975    4654.575630   40547.165966   95349.981092\n",
      "std             13329.295173    2022.508892   23509.868457   38353.638258\n",
      "min             10004.510000    1649.000000     161.000000   24853.000000\n",
      "25%             17799.760000    2822.000000   24857.500000   69255.750000\n",
      "50%             18585.960000    3805.000000   36322.500000   88437.500000\n",
      "75%             32726.550000    5344.000000   53332.750000  112365.000000\n",
      "max             74452.950000   12297.000000  112750.000000  250001.000000\n"
     ]
    }
   ],
   "source": [
    "print(\"Descriptive Statistics:\")\n",
    "print(df_integrated[['residential_mwh_sales', 'num_customers', 'population', 'median_income']].describe())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 37,
   "id": "873755c3",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 1400x1000 with 4 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "fig, axes = plt.subplots(2, 2, figsize=(14, 10))\n",
    "\n",
    "df_integrated.boxplot(column='residential_mwh_sales', by='city', ax=axes[0, 0])\n",
    "axes[0, 0].set_title('Residential MWh Sales by City')\n",
    "axes[0, 0].set_xlabel('City'); axes[0, 0].set_ylabel('MWh')\n",
    "\n",
    "df_integrated.boxplot(column='population', by='city', ax=axes[0, 1])\n",
    "axes[0, 1].set_title('Population by City')\n",
    "axes[0, 1].set_xlabel('City'); axes[0, 1].set_ylabel('Population')\n",
    "\n",
    "df_integrated.boxplot(column='median_income', by='city', ax=axes[1, 0])\n",
    "axes[1, 0].set_title('Median Income by City')\n",
    "axes[1, 0].set_xlabel('City'); axes[1, 0].set_ylabel('Income ($)')\n",
    "\n",
    "df_integrated.boxplot(column='num_customers', by='city', ax=axes[1, 1])\n",
    "axes[1, 1].set_title('Number of Customers by City')\n",
    "axes[1, 1].set_xlabel('City'); axes[1, 1].set_ylabel('Customers')\n",
    "\n",
    "plt.suptitle('')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 38,
   "id": "bdec8d57",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 1200x600 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "fig, ax = plt.subplots(figsize=(12, 6))\n",
    "for city in ['NYC', 'LA']:\n",
    "    mask = df_integrated['city'] == city\n",
    "    ax.scatter(df_integrated[mask]['median_income'],\n",
    "               df_integrated[mask]['residential_mwh_sales'],\n",
    "               label=city, alpha=0.6, s=100)\n",
    "ax.set_xlabel('Median Household Income ($)', fontsize=12)\n",
    "ax.set_ylabel('Residential MWh Sales', fontsize=12)\n",
    "ax.set_title('Electricity Sales vs Income by City', fontsize=14)\n",
    "ax.legend(); ax.grid(True, alpha=0.3)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 39,
   "id": "ecee0d2f",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 800x600 with 2 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "corr_cols = ['residential_mwh_sales', 'num_customers', 'population', 'median_income']\n",
    "corr_matrix = df_integrated[corr_cols].corr()\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(8, 6))\n",
    "sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)\n",
    "ax.set_title('Correlation Matrix: Key Variables', fontsize=14)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "11b47d3a",
   "metadata": {},
   "source": [
    "---\n",
    "\n",
    "## Section 4: Feature Engineering"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 40,
   "id": "cf552336",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "[Features] Dropped 1 rows with invalid/missing features\n",
      "[Features] Engineered 5 features for 475 ZIPs:\n",
      "  - electricity_per_customer\n",
      "  - electricity_per_capita\n",
      "  - renter_occupancy_rate\n",
      "  - housing_age\n",
      "  - income_log\n",
      "\n",
      "Feature Summary Statistics:\n",
      "       electricity_per_customer  electricity_per_capita  \\\n",
      "count                475.000000              475.000000   \n",
      "mean                   5.787014                1.486311   \n",
      "std                    0.912029                4.274758   \n",
      "min                    4.677992                0.157869   \n",
      "25%                    4.677992                0.408393   \n",
      "50%                    6.123980                0.657570   \n",
      "75%                    6.586095                1.247281   \n",
      "max                    9.147492               62.160401   \n",
      "\n",
      "       renter_occupancy_rate  housing_age  income_log  \n",
      "count             475.000000   475.000000  475.000000  \n",
      "mean                0.546335    61.553684   11.386031  \n",
      "std                 0.223447    14.534215    0.394602  \n",
      "min                 0.060450    12.000000   10.120774  \n",
      "25%                 0.356841    54.000000   11.144835  \n",
      "50%                 0.541369    63.000000   11.388835  \n",
      "75%                 0.711117    70.000000   11.629098  \n",
      "max                 1.000000    84.000000   12.429224  \n",
      "\n",
      "Dataset with engineered features:\n",
      "        ZIP  residential_mwh_sales  num_customers state  population  \\\n",
      "2193  10001               17799.76           3805    NY       27004   \n",
      "2194  10002               17799.76           3805    NY       76518   \n",
      "2195  10003               17799.76           3805    NY       53877   \n",
      "2196  10004               17799.76           3805    NY        4579   \n",
      "2197  10005               17799.76           3805    NY        8801   \n",
      "\n",
      "      median_income  median_year_structure_built  total_occupied_units  \\\n",
      "2193       106509.0                       1969.0                 14375   \n",
      "2194        43362.0                       1952.0                 36028   \n",
      "2195       152863.0                       1938.0                 24987   \n",
      "2196       232543.0                       1938.0                  2123   \n",
      "2197       189886.0                       1938.0                  4881   \n",
      "\n",
      "      renter_occupied_units city  electricity_per_customer  \\\n",
      "2193                  10953  NYC                  4.677992   \n",
      "2194                  29804  NYC                  4.677992   \n",
      "2195                  16060  NYC                  4.677992   \n",
      "2196                   1230  NYC                  4.677992   \n",
      "2197                   4067  NYC                  4.677992   \n",
      "\n",
      "      electricity_per_capita  renter_occupancy_rate  housing_age  income_log  \n",
      "2193                0.659153               0.761948         53.0   11.575994  \n",
      "2194                0.232622               0.827245         70.0   10.677362  \n",
      "2195                0.330378               0.642734         84.0   11.937304  \n",
      "2196                3.887259               0.579369         84.0   12.356835  \n",
      "2197                2.022470               0.833231         84.0   12.154184  \n"
     ]
    }
   ],
   "source": [
    "df_features = engineer_features(df_integrated)\n",
    "print(\"\\nDataset with engineered features:\")\n",
    "print(df_features.head())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 41,
   "id": "76fca937",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 1500x800 with 6 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "feature_cols = [\n",
    "    'electricity_per_customer', 'electricity_per_capita',\n",
    "    'renter_occupancy_rate', 'housing_age', 'income_log'\n",
    "]\n",
    "\n",
    "fig, axes = plt.subplots(2, 3, figsize=(15, 8))\n",
    "axes = axes.flatten()\n",
    "\n",
    "for i, col in enumerate(feature_cols):\n",
    "    axes[i].hist(df_features[col], bins=30, alpha=0.7, edgecolor='black')\n",
    "    axes[i].set_xlabel(col); axes[i].set_ylabel('Frequency')\n",
    "    axes[i].set_title(f'Distribution of {col}')\n",
    "    axes[i].grid(True, alpha=0.3)\n",
    "\n",
    "axes[-1].axis('off')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "fbee6464",
   "metadata": {},
   "source": [
    "---\n",
    "\n",
    "## Section 5: PCA Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 42,
   "id": "e27aa446",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Feature matrix shape: (475, 5)\n",
      "Columns: ['electricity_per_customer', 'electricity_per_capita', 'renter_occupancy_rate', 'housing_age', 'income_log']\n"
     ]
    }
   ],
   "source": [
    "feature_matrix = get_feature_matrix(df_features)\n",
    "print(f\"Feature matrix shape: {feature_matrix.shape}\")\n",
    "print(f\"Columns: {feature_matrix.columns.tolist()}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 43,
   "id": "bd829d58",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "[Standardize] Standardized 5 features for 475 ZIPs\n",
      "Standardized features shape: (475, 5)\n"
     ]
    }
   ],
   "source": [
    "scaler, std_features = standardize_features(feature_matrix)\n",
    "print(f\"Standardized features shape: {std_features.shape}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 44,
   "id": "72e16f84",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "[PCA] Selected 4 components explaining 93.03% variance\n",
      "[PCA] Explained variance per component: [0.40247779 0.19488106 0.18304054 0.14987354]\n",
      "\n",
      "PCA Results:\n",
      "Number of components: 4\n",
      "Explained variance ratio: [0.40247779 0.19488106 0.18304054 0.14987354]\n",
      "Cumulative variance: [0.40247779 0.59735886 0.7803994  0.93027294]\n"
     ]
    }
   ],
   "source": [
    "pca_data = apply_pca(std_features, variance_threshold=0.85)\n",
    "\n",
    "print(f\"\\nPCA Results:\")\n",
    "print(f\"Number of components: {pca_data['n_components']}\")\n",
    "print(f\"Explained variance ratio: {pca_data['explained_variance_ratio']}\")\n",
    "print(f\"Cumulative variance: {pca_data['cumulative_variance']}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 45,
   "id": "2b49a211",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 1400x500 with 2 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "axes[0].bar(range(1, len(pca_data['explained_variance_ratio']) + 1),\n",
    "            pca_data['explained_variance_ratio'], alpha=0.7, edgecolor='black')\n",
    "axes[0].set_xlabel('Principal Component')\n",
    "axes[0].set_ylabel('Explained Variance Ratio')\n",
    "axes[0].set_title('Scree Plot')\n",
    "axes[0].grid(True, alpha=0.3)\n",
    "\n",
    "axes[1].plot(range(1, len(pca_data['cumulative_variance']) + 1),\n",
    "             pca_data['cumulative_variance'], marker='o', linestyle='--', linewidth=2)\n",
    "axes[1].axhline(y=0.85, color='r', linestyle='--', label='85% Threshold')\n",
    "axes[1].set_xlabel('Number of Components')\n",
    "axes[1].set_ylabel('Cumulative Explained Variance')\n",
    "axes[1].set_title('Cumulative Explained Variance')\n",
    "axes[1].legend()\n",
    "axes[1].grid(True, alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "algo_compare_md",
   "metadata": {},
   "source": [
    "---\n",
    "\n",
    "## Section 6: Algorithm Comparison\n",
    "\n",
    "Rather than committing to a single clustering algorithm upfront, we run three candidates on the PCA-transformed data and select the best one using three complementary metrics:\n",
    "\n",
    "| Metric | What it measures | Goal |\n",
    "|--------|-----------------|------|\n",
    "| **Silhouette Score** | Cohesion + separation per point | Maximize (range: -1 to 1) |\n",
    "| **Davies-Bouldin Index** | Average cluster similarity ratio | Minimize (lower = better separated) |\n",
    "| **Calinski-Harabasz Index** | Between-cluster vs within-cluster variance | Maximize |\n",
    "\n",
    "**Candidates:**\n",
    "- **K-Means** — fast baseline; assumes spherical, equal-sized clusters\n",
    "- **Agglomerative Hierarchical (Ward)** — handles unequal cluster sizes; dendrogram gives structural insight\n",
    "- **DBSCAN** — density-based; identifies outliers as noise; no k required"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 46,
   "id": "run_hierarchical",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Running Agglomerative Hierarchical Clustering...\n",
      "[Hierarchical] k=2: silhouette = 0.2576\n",
      "[Hierarchical] k=3: silhouette = 0.2464\n",
      "[Hierarchical] k=4: silhouette = 0.2545\n",
      "[Hierarchical] k=5: silhouette = 0.2675\n",
      "[Hierarchical] k=6: silhouette = 0.2697\n",
      "[Hierarchical] k=7: silhouette = 0.2598\n",
      "[Hierarchical] Optimal k = 6 (silhouette = 0.2697)\n"
     ]
    }
   ],
   "source": [
    "# ── Algorithm A: Agglomerative Hierarchical Clustering ──────────────────────\n",
    "print(\"Running Agglomerative Hierarchical Clustering...\")\n",
    "hierarchical_result = apply_clustering(pca_data, k_min=2, k_max=7)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 47,
   "id": "run_kmeans",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Running K-Means Clustering...\n",
      "[K-Means] k=2: silhouette = 0.3019, inertia = 1590.46\n",
      "[K-Means] k=3: silhouette = 0.3181, inertia = 1223.43\n",
      "[K-Means] k=4: silhouette = 0.3132, inertia = 974.92\n",
      "[K-Means] k=5: silhouette = 0.3074, inertia = 822.15\n",
      "[K-Means] k=6: silhouette = 0.3093, inertia = 691.42\n",
      "[K-Means] k=7: silhouette = 0.3028, inertia = 627.59\n",
      "[K-Means] Optimal k = 3 (silhouette = 0.3181)\n"
     ]
    },
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 800x400 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "# ── Algorithm B: K-Means ────────────────────────────────────────────────────\n",
    "print(\"Running K-Means Clustering...\")\n",
    "kmeans_result = apply_kmeans(pca_data, k_min=2, k_max=7)\n",
    "\n",
    "# Elbow plot (inertia)\n",
    "fig, ax = plt.subplots(figsize=(8, 4))\n",
    "ks = sorted(kmeans_result['inertia_scores'].keys())\n",
    "ax.plot(ks, [kmeans_result['inertia_scores'][k] for k in ks], marker='o', linewidth=2)\n",
    "ax.set_xlabel('Number of Clusters (k)')\n",
    "ax.set_ylabel('Inertia (Within-Cluster SSE)')\n",
    "ax.set_title('K-Means Elbow Plot')\n",
    "ax.grid(True, alpha=0.3)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 48,
   "id": "run_dbscan",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Running DBSCAN...\n",
      "[DBSCAN] eps=0.3: 17 clusters, 311 noise points, silhouette = 0.3582\n",
      "[DBSCAN] eps=0.5: 5 clusters, 102 noise points, silhouette = 0.2907\n",
      "[DBSCAN] eps=0.7: 2 clusters, 48 noise points, silhouette = 0.4679\n",
      "[DBSCAN] eps=1.0: 2 clusters, 23 noise points, silhouette = 0.4554\n",
      "[DBSCAN] eps=1.5: 2 clusters, 10 noise points, silhouette = 0.4426\n",
      "[DBSCAN] eps=2.0: 2 clusters, 6 noise points, silhouette = 0.4375\n",
      "[DBSCAN] Best eps=0.7: 2 clusters, silhouette = 0.4679\n"
     ]
    }
   ],
   "source": [
    "# ── Algorithm C: DBSCAN ─────────────────────────────────────────────────────\n",
    "print(\"Running DBSCAN...\")\n",
    "dbscan_result = apply_dbscan(pca_data, eps_values=[0.3, 0.5, 0.7, 1.0, 1.5, 2.0], min_samples=5)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 49,
   "id": "compare_table",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "======================================================================\n",
      "ALGORITHM COMPARISON SUMMARY\n",
      "======================================================================\n",
      "                    Silhouette ↑  Davies-Bouldin ↓  Calinski-Harabasz ↑  n_clusters  n_noise\n",
      "Algorithm                                                                                   \n",
      "Hierarchical (k=6)        0.2697            1.0045               181.33           6        0\n",
      "K-Means (k=3)             0.3181            0.9764               190.19           3        0\n",
      "DBSCAN (eps=0.7)          0.4679            0.4821                28.87           2       48\n",
      "\n",
      "Best Silhouette  : DBSCAN (eps=0.7)\n",
      "Best Davies-Bouldin: DBSCAN (eps=0.7)\n",
      "Best Calinski-Harabasz: K-Means (k=3)\n"
     ]
    },
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>Silhouette ↑</th>\n",
       "      <th>Davies-Bouldin ↓</th>\n",
       "      <th>Calinski-Harabasz ↑</th>\n",
       "      <th>n_clusters</th>\n",
       "      <th>n_noise</th>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Algorithm</th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>Hierarchical (k=6)</th>\n",
       "      <td>0.2697</td>\n",
       "      <td>1.0045</td>\n",
       "      <td>181.33</td>\n",
       "      <td>6</td>\n",
       "      <td>0</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>K-Means (k=3)</th>\n",
       "      <td>0.3181</td>\n",
       "      <td>0.9764</td>\n",
       "      <td>190.19</td>\n",
       "      <td>3</td>\n",
       "      <td>0</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>DBSCAN (eps=0.7)</th>\n",
       "      <td>0.4679</td>\n",
       "      <td>0.4821</td>\n",
       "      <td>28.87</td>\n",
       "      <td>2</td>\n",
       "      <td>48</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "                    Silhouette ↑  Davies-Bouldin ↓  Calinski-Harabasz ↑  \\\n",
       "Algorithm                                                                 \n",
       "Hierarchical (k=6)        0.2697            1.0045               181.33   \n",
       "K-Means (k=3)             0.3181            0.9764               190.19   \n",
       "DBSCAN (eps=0.7)          0.4679            0.4821                28.87   \n",
       "\n",
       "                    n_clusters  n_noise  \n",
       "Algorithm                                \n",
       "Hierarchical (k=6)           6        0  \n",
       "K-Means (k=3)                3        0  \n",
       "DBSCAN (eps=0.7)             2       48  "
      ]
     },
     "execution_count": 49,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "# ── Comparison Table ─────────────────────────────────────────────────────────\n",
    "comparison_df = compare_algorithms(pca_data, hierarchical_result, kmeans_result, dbscan_result)\n",
    "comparison_df"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 50,
   "id": "compare_plot",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 1600x500 with 3 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "# ── Visualize metric comparison ──────────────────────────────────────────────\n",
    "fig, axes = plt.subplots(1, 3, figsize=(16, 5))\n",
    "\n",
    "algos = comparison_df.index.tolist()\n",
    "colors = ['steelblue', 'coral', 'mediumseagreen'][:len(algos)]\n",
    "\n",
    "axes[0].bar(algos, comparison_df['Silhouette ↑'], color=colors, edgecolor='black', alpha=0.8)\n",
    "axes[0].set_title('Silhouette Score (higher = better)')\n",
    "axes[0].set_ylabel('Score'); axes[0].tick_params(axis='x', rotation=15)\n",
    "axes[0].grid(True, alpha=0.3, axis='y')\n",
    "\n",
    "axes[1].bar(algos, comparison_df['Davies-Bouldin ↓'], color=colors, edgecolor='black', alpha=0.8)\n",
    "axes[1].set_title('Davies-Bouldin Index (lower = better)')\n",
    "axes[1].set_ylabel('Score'); axes[1].tick_params(axis='x', rotation=15)\n",
    "axes[1].grid(True, alpha=0.3, axis='y')\n",
    "\n",
    "axes[2].bar(algos, comparison_df['Calinski-Harabasz ↑'], color=colors, edgecolor='black', alpha=0.8)\n",
    "axes[2].set_title('Calinski-Harabasz Index (higher = better)')\n",
    "axes[2].set_ylabel('Score'); axes[2].tick_params(axis='x', rotation=15)\n",
    "axes[2].grid(True, alpha=0.3, axis='y')\n",
    "\n",
    "plt.suptitle('Clustering Algorithm Comparison', fontsize=14, y=1.02)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 51,
   "id": "select_algorithm",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Best Silhouette:        DBSCAN (eps=0.7)\n",
      "Best Davies-Bouldin:    DBSCAN (eps=0.7)\n",
      "Best Calinski-Harabasz: K-Means (k=3)\n",
      "\n",
      ">>> Selected Algorithm: DBSCAN (eps=0.7) <<<\n",
      "Final algorithm: DBSCAN\n",
      "Number of clusters: 2\n",
      "Silhouette score:   0.4679\n"
     ]
    }
   ],
   "source": [
    "# ── Select winning algorithm ─────────────────────────────────────────────────\n",
    "# Decision: choose based on best combined metric profile\n",
    "# Tiebreak rules: Silhouette > Davies-Bouldin > Calinski-Harabasz\n",
    "\n",
    "best_sil_algo = comparison_df['Silhouette ↑'].idxmax()\n",
    "best_db_algo  = comparison_df['Davies-Bouldin ↓'].idxmin()\n",
    "best_ch_algo  = comparison_df['Calinski-Harabasz ↑'].idxmax()\n",
    "\n",
    "print(f\"Best Silhouette:        {best_sil_algo}\")\n",
    "print(f\"Best Davies-Bouldin:    {best_db_algo}\")\n",
    "print(f\"Best Calinski-Harabasz: {best_ch_algo}\")\n",
    "print()\n",
    "\n",
    "# Pick whichever algorithm wins the most metrics\n",
    "from collections import Counter\n",
    "votes = Counter([best_sil_algo, best_db_algo, best_ch_algo])\n",
    "winner = votes.most_common(1)[0][0]\n",
    "print(f\">>> Selected Algorithm: {winner} <<<\")\n",
    "\n",
    "# Map winner name back to result dict\n",
    "if 'Hierarchical' in winner:\n",
    "    selected_result = hierarchical_result\n",
    "    selected_name = 'Agglomerative Hierarchical (Ward)'\n",
    "elif 'K-Means' in winner:\n",
    "    selected_result = kmeans_result\n",
    "    selected_name = 'K-Means'\n",
    "else:\n",
    "    selected_result = dbscan_result\n",
    "    selected_name = 'DBSCAN'\n",
    "\n",
    "print(f\"Final algorithm: {selected_name}\")\n",
    "print(f\"Number of clusters: {selected_result.get('optimal_k', selected_result.get('n_clusters'))}\")\n",
    "print(f\"Silhouette score:   {selected_result['best_silhouette']:.4f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "6432bad8",
   "metadata": {},
   "source": [
    "---\n",
    "\n",
    "## Section 7: Final Clustering — Dendrogram and Silhouette Plot\n",
    "\n",
    "Using the winning algorithm selected above."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 52,
   "id": "8ae573e4",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 1000x500 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "# Silhouette sweep for the selected algorithm\n",
    "fig, ax = plt.subplots(figsize=(10, 5))\n",
    "\n",
    "k_values = sorted(selected_result['silhouette_scores'].keys())\n",
    "scores = [selected_result['silhouette_scores'][k] for k in k_values]\n",
    "bars = ax.bar(k_values, scores, alpha=0.7, edgecolor='black')\n",
    "\n",
    "optimal_k = selected_result.get('optimal_k', selected_result.get('n_clusters'))\n",
    "if optimal_k in k_values:\n",
    "    bars[k_values.index(optimal_k)].set_color('red')\n",
    "\n",
    "ax.set_xlabel('Number of Clusters (k)')\n",
    "ax.set_ylabel('Silhouette Score')\n",
    "ax.set_title(f'Silhouette Score vs k — {selected_name} (optimal k={optimal_k})')\n",
    "ax.set_xticks(k_values)\n",
    "ax.grid(True, alpha=0.3, axis='y')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 53,
   "id": "83882861",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 1400x800 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "# Dendrogram (subsample 200 ZIPs for readability)\n",
    "sample_size = min(200, len(pca_data['transformed']))\n",
    "np.random.seed(42)\n",
    "sample_idx = np.random.choice(len(pca_data['transformed']), sample_size, replace=False)\n",
    "sample_features = pca_data['transformed'][sample_idx]\n",
    "\n",
    "Z = linkage(sample_features, method='ward')\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(14, 8))\n",
    "dendrogram(Z, ax=ax, leaf_rotation=90, leaf_font_size=6, no_labels=True)\n",
    "ax.set_xlabel('ZIP Code (sampled)')\n",
    "ax.set_ylabel('Ward Distance')\n",
    "ax.set_title(f'Hierarchical Clustering Dendrogram (n={sample_size} sampled ZIPs)')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "f0137247",
   "metadata": {},
   "source": [
    "---\n",
    "\n",
    "## Section 8: Cluster Profiling and NYC vs LA Comparison"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 54,
   "id": "a1b2c3d4",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "[Evaluation] Cluster Profiles (means):\n",
      "         electricity_per_customer  electricity_per_capita  \\\n",
      "cluster                                                     \n",
      "-1                       5.873793                6.865431   \n",
      " 0                       5.737327                0.868344   \n",
      " 1                       9.147492                2.003220   \n",
      "\n",
      "         renter_occupancy_rate  housing_age  income_log  median_income  \n",
      "cluster                                                                 \n",
      "-1                    0.569016    48.479167   11.633699  129377.895833  \n",
      " 0                    0.547982    62.981043   11.351302   90421.497630  \n",
      " 1                    0.189618    66.600000   11.939574  153715.800000  \n",
      "\n",
      "[Evaluation] Cluster distribution by city:\n",
      "city      LA  NYC  All\n",
      "cluster               \n",
      "-1        29   19   48\n",
      "0        262  160  422\n",
      "1          0    5    5\n",
      "All      291  184  475\n",
      "\n",
      "Cluster Sizes:\n",
      "cluster\n",
      "-1     48\n",
      " 0    422\n",
      " 1      5\n",
      "Name: count, dtype: int64\n"
     ]
    }
   ],
   "source": [
    "eval_data = evaluate_clustering(df_features, pca_data, selected_result)\n",
    "df_clustered = eval_data['df_clustered']\n",
    "\n",
    "print(\"\\nCluster Sizes:\")\n",
    "print(df_clustered['cluster'].value_counts().sort_index())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 55,
   "id": "67ac3f50",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 1600x600 with 3 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "# PCA scatter colored by cluster\n",
    "fig, axes = plt.subplots(1, 2, figsize=(16, 6))\n",
    "\n",
    "# Left: colored by cluster\n",
    "scatter = axes[0].scatter(\n",
    "    pca_data['transformed'][:, 0],\n",
    "    pca_data['transformed'][:, 1],\n",
    "    c=selected_result['labels'], cmap='tab10',\n",
    "    s=80, alpha=0.7, edgecolors='black', linewidths=0.3\n",
    ")\n",
    "axes[0].set_xlabel(f'PC1 ({pca_data[\"explained_variance_ratio\"][0]:.1%} variance)')\n",
    "axes[0].set_ylabel(f'PC2 ({pca_data[\"explained_variance_ratio\"][1]:.1%} variance)')\n",
    "axes[0].set_title(f'PCA Space — Colored by Cluster ({selected_name})')\n",
    "plt.colorbar(scatter, ax=axes[0], label='Cluster')\n",
    "\n",
    "# Right: colored by city\n",
    "city_colors = {'NYC': 'steelblue', 'LA': 'coral'}\n",
    "for city, color in city_colors.items():\n",
    "    mask = df_clustered['city'] == city\n",
    "    axes[1].scatter(\n",
    "        pca_data['transformed'][mask.values, 0],\n",
    "        pca_data['transformed'][mask.values, 1],\n",
    "        c=color, label=city, s=80, alpha=0.6, edgecolors='black', linewidths=0.3\n",
    "    )\n",
    "axes[1].set_xlabel(f'PC1 ({pca_data[\"explained_variance_ratio\"][0]:.1%} variance)')\n",
    "axes[1].set_ylabel(f'PC2 ({pca_data[\"explained_variance_ratio\"][1]:.1%} variance)')\n",
    "axes[1].set_title('PCA Space — Colored by City')\n",
    "axes[1].legend()\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 56,
   "id": "dfdeb420",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 1400x500 with 2 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "# NYC vs LA cluster distribution\n",
    "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "cluster_city_counts = pd.crosstab(df_clustered['cluster'], df_clustered['city'])\n",
    "cluster_city_pct = cluster_city_counts.div(cluster_city_counts.sum(axis=1), axis=0) * 100\n",
    "\n",
    "cluster_city_counts.plot(kind='bar', ax=axes[0], colormap='Set1', edgecolor='black')\n",
    "axes[0].set_title('ZIP Count per Cluster by City')\n",
    "axes[0].set_xlabel('Cluster'); axes[0].set_ylabel('Number of ZIPs')\n",
    "axes[0].tick_params(axis='x', rotation=0)\n",
    "axes[0].grid(True, alpha=0.3, axis='y')\n",
    "\n",
    "cluster_city_pct.plot(kind='bar', ax=axes[1], colormap='Set1', edgecolor='black')\n",
    "axes[1].set_title('Cluster Composition (%) by City')\n",
    "axes[1].set_xlabel('Cluster'); axes[1].set_ylabel('% of ZIPs in Cluster')\n",
    "axes[1].tick_params(axis='x', rotation=0)\n",
    "axes[1].grid(True, alpha=0.3, axis='y')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 57,
   "id": "f9662ba0",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "================================================================================\n",
      "CLUSTER SUMMARY STATISTICS\n",
      "================================================================================\n",
      "\n",
      "Noise points (DBSCAN): 48 ZIPs\n",
      "\n",
      "Cluster 0: 422 ZIPs (160 NYC, 262 LA)\n",
      "--------------------------------------------------------------------------------\n",
      "      electricity_per_customer  electricity_per_capita  median_income  \\\n",
      "mean                      5.74                    0.87       90421.50   \n",
      "std                       0.82                    0.81       31492.29   \n",
      "\n",
      "      population  \n",
      "mean    43871.53  \n",
      "std     22526.24  \n",
      "\n",
      "Cluster 1: 5 ZIPs (5 NYC, 0 LA)\n",
      "--------------------------------------------------------------------------------\n",
      "      electricity_per_customer  electricity_per_capita  median_income  \\\n",
      "mean                      9.15                    2.00      153715.80   \n",
      "std                       0.00                    1.23       13990.53   \n",
      "\n",
      "      population  \n",
      "mean    16524.00  \n",
      "std     11527.36  \n"
     ]
    }
   ],
   "source": [
    "print(\"\\n\" + \"=\"*80)\n",
    "print(\"CLUSTER SUMMARY STATISTICS\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "summary_cols = ['electricity_per_customer', 'electricity_per_capita', 'median_income', 'population']\n",
    "\n",
    "for cluster_id in sorted(df_clustered['cluster'].unique()):\n",
    "    if cluster_id == -1:\n",
    "        print(f\"\\nNoise points (DBSCAN): {(df_clustered['cluster'] == -1).sum()} ZIPs\")\n",
    "        continue\n",
    "    cluster_data = df_clustered[df_clustered['cluster'] == cluster_id]\n",
    "    n_nyc = (cluster_data['city'] == 'NYC').sum()\n",
    "    n_la  = (cluster_data['city'] == 'LA').sum()\n",
    "    print(f\"\\nCluster {cluster_id}: {len(cluster_data)} ZIPs ({n_nyc} NYC, {n_la} LA)\")\n",
    "    print(\"-\" * 80)\n",
    "    print(cluster_data[summary_cols].describe().loc[['mean', 'std']].round(2))"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "27f771f5",
   "metadata": {},
   "source": [
    "---\n",
    "\n",
    "## Conclusion\n",
    "\n",
    "This notebook evaluated three clustering algorithms (K-Means, Agglomerative Hierarchical with Ward linkage, and DBSCAN) using three complementary metrics (Silhouette Score, Davies-Bouldin Index, Calinski-Harabasz Index). The winning algorithm was selected based on the best combined metric profile.\n",
    "\n",
    "The results reveal distinct neighborhood energy consumption archetypes across NYC and LA ZIP codes, with structural differences tied to income, housing age, and renter occupancy rates. Two clusters emerge as nearly city-exclusive, confirming that NYC and LA occupy fundamentally different positions in the socio-economic energy-use landscape."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.11.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}
Apollo
